# Wikipedia Search Agent

In [ ]:
def search_wikipedia(query: str) -> dict:
    """Search API - Find pages related to a topic.
    
    Args:
        query (str): The search query string.
    Returns:
        dict: The JSON response from the Wikipedia Search API.
    """

    search_url = f"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}"
    headers = {
        "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
    }
    response = requests.get(search_url, headers=headers)

    return response.json()

In [41]:
result = search_wikipedia("capybara")

search_pages = result["query"]["search"]

# HOMEWORK QUESTION 1: How many search results did we get for the query "capybara"?
len(search_pages)

10

In [ ]:
# HOMEWORK QUESTION 2: How many of the search results have "capybara" in the title?
len([page for page in search_pages if "capybara" in page["title"].lower()])

5

In [24]:
def get_page(title: str) -> str:
    """Page API - Get the raw content of a Wikipedia page.
    
    Args:
        title (str): The title of the Wikipedia page to retrieve.
    Returns:
        str: The raw content of the Wikipedia page.
    """

    page_url = f"https://en.wikipedia.org/w/index.php?title={title}&action=raw"
    headers = {
        "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
    }
    response = requests.get(page_url, headers=headers)

    return response.text

In [ ]:
# HOMEWORK QUESTION 3: How many characters are in the raw content of the "Capybara" Wikipedia page?
len(get_page("Capybara"))

36946

In [28]:
class WikipediaSearchTools:
    """Provides tools for searching Wikipedia and retrieving page content."""

    def __init__(self) -> None:
        """Initializes the WikipediaSearchTools instance."""

        self.headers = {
            "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
        }

    def search_wikipedia(self, query: str) -> dict:
        """Search API - Finds pages related to a topic.
        
        Args:
            query (str): The search query string.
        Returns:
            dict: The JSON response from the Wikipedia Search API.
        """

        search_url = f"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}"
        response = requests.get(search_url, headers=self.headers)

        return response.json()
    
    def get_page(self, title: str) -> str:
        """Page API - Gets the raw content of a Wikipedia page.
        
        Args:
            title (str): The title of the Wikipedia page to retrieve.
        Returns:
            str: The raw content of the Wikipedia page.
        """

        page_url = f"https://en.wikipedia.org/w/index.php?title={title}&action=raw"
        response = requests.get(page_url, headers=self.headers)

        return response.text

In [29]:
wikipedia_search_tools = WikipediaSearchTools()

tools = [wikipedia_search_tools.search_wikipedia, wikipedia_search_tools.get_page]

In [ ]:
from pydantic_ai import Agent

instructions = """
You are a helpful assistant that can search Wikipedia, retrieve relevant page content and answer questions based on that content. 

Answer questions by following these steps:
1. Use the search_wikipedia tool to find relevant pages for the user's query.
2. Use the get_page tool to retrieve the raw content of relevant pages.
3. Use the retrieved content to answer the user's question as accurately as possible.

Make 3 attempts to find the answer. If you cannot find the answer after 3 attempts, respond with "I don't know".
"""

search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [31]:
query = 'How many people live in Amsterdam?'

await search_agent.run(query)

AgentRunResult(output='As of June 2024, the population of Amsterdam is approximately **933,680**. In terms of urban and metropolitan area populations, Amsterdam has about **1,477,213** in the urban area and around **2,480,394** in the metropolitan area.')

In [ ]:
# HOMEWORK QUESTION 4: What is your agent setup?
# I used Pydantic AI + OpenAI gpt-4o-mini model.

In [34]:
query = "What is this page about? https://en.wikipedia.org/wiki/Capybara"

result = await search_agent.run(query)

In [ ]:
from pydantic_ai.agent import AgentRunResult

def pprint_agent_response(result: AgentRunResult) -> None:
    """Pretty prints the agent's response, showing the kind of each message and the content of each part.
    
    Args:
        result (AgentRunResult): The result object returned by the agent after running a query.
    
    Returns:
        None
    """

    messages = result.all_messages()
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

In [ ]:
# HOMEWORK QUESTION 5: What does agent answer for the above query?
pprint_agent_response(result)

request
USER: What is this page about? https://en.wikipedia.org/wiki/Capybara

response
TOOL CALL: get_page {"title":"Capybara"}

request
TOOL RETURN: get_page

response
The Wikipedia page on the **capybara** describes it as the largest living rodent, scientifically known as *Hydrochoerus hydrochaeris*. It is native to all South American countries except Chile and is a semiaquatic herbivore often found near water bodies such as rivers and lakes. 

**Key points from the page include:**

- **Physical Characteristics**: Capybaras have a heavy body, short head, reddish-brown fur on top that fades to yellowish-brown underneath, and are generally social animals that live in groups.
- **Habitat and Diet**: They inhabit savannas and dense forests, primarily feeding on grasses and aquatic plants.
- **Social Behavior**: These animals are social, often forming groups of 10 to 20, but can be seen in larger groups. They communicate through a variety of vocalizations including barks and whistles.
- 

In [40]:
query = "What are the main threats to capybara populations?"

result = await search_agent.run(query)

# HOMEWORK QUESTION 6: How many total tool calls (search + fetch combined) did your agent make?
pprint_agent_response(result)

request
USER: What are the main threats to capybara populations?

response
TOOL CALL: search_wikipedia {"query":"capybara threats"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: search_wikipedia {"query":"capybara conservation threats"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: search_wikipedia {"query":"capybara endangered species threats"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: get_page {"title": "Pantanal"}
TOOL CALL: get_page {"title": "Jaguar"}
TOOL CALL: get_page {"title": "Aquatic mammal"}

request
TOOL RETURN: get_page
TOOL RETURN: get_page
TOOL RETURN: get_page

response
The main threats to capybara populations include:

1. **Habitat Loss**: As with many wildlife species, the loss of their natural habitats due to urban development, agriculture, and deforestation poses a significant threat. Capybaras rely on wetlands and riverine areas that are increasingly being converted for other uses.

2. **Hunting and Poaching**: Capybar